In [84]:
import pandas as pd
import numpy as np
import pandas as pd
import warnings
import os
import threadpoolctl

warnings.filterwarnings('ignore')

# For vector models optimized ranking
os.environ["OPENBLAS_NUM_THREADS"] = "1"
threadpoolctl.threadpool_limits(1, "blas");

In [85]:
columns = ['user_id','item_id','weight', 'datetime']

In [5]:
df = pd.read_csv('./04-metrika-pipeline-dataset-rt.csv')
df = df[columns]
df.sample(3)

,user_id,item_id,weight,datetime
34072,169633171077598653,Hearts of Iron IV: By Blood Alone,1,2023-10-03 14:11:02
221110,1728017688466175763,Captain Tsubasa: Rise of New Champions - Ultim...,1,2024-10-04 07:54:38
363204,1745465342244797720,Ready or Not,1,2025-04-23 19:29:33


In [111]:
df.shape

(448530, 4)

In [6]:
df["user_id"].nunique(), df["item_id"].nunique()

(207838, 8044)

In [7]:
df["datetime"].min(), df["datetime"].max()

('2023-09-19 08:18:53', '2025-10-11 23:57:15')

In [50]:
from implicit.nearest_neighbours import TFIDFRecommender, BM25Recommender
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

from lightfm import LightFM

from rectools.dataset import Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity, NDCG, calc_metrics
from rectools.model_selection import TimeRangeSplitter, LastNSplitter, cross_validate

from rectools.models import (
    ImplicitALSWrapperModel, 
    ImplicitBPRWrapperModel, 
    LightFMWrapperModel, 
    PureSVDModel, 
    EASEModel,
    ImplicitItemKNNWrapperModel, 
    RandomModel, 
    PopularModel
)

In [18]:
dataset = Dataset.construct(df)

In [20]:
k = 5

In [51]:
# Take few simple models to compare
models = {
    "random": RandomModel(random_state=42),
    "popular": PopularModel(),
    "most_rated": PopularModel(popularity="sum_weight"),
    "knn_tfidf_k=5": ImplicitItemKNNWrapperModel(model=TFIDFRecommender(K=k)),
    "knn_bm25_k=5": ImplicitItemKNNWrapperModel(model=BM25Recommender(K=k, K1=0.05, B=0.1)),
    "als": ImplicitALSWrapperModel(
        AlternatingLeastSquares(
            factors=10,  # latent embeddings size
            regularization=0.1, 
            iterations=10,
            alpha=50,  # confidence multiplier for non-zero entries in interactions
            random_state=42,
        )
    ),
    'brp': ImplicitBPRWrapperModel(
        BayesianPersonalizedRanking(
            factors=10,  # latent embeddings size
            regularization=0.1, 
            iterations=10,
            random_state=42,
        ),
    ),
    'lightfm': LightFMWrapperModel(LightFM(no_components=10, loss="bpr", random_state=42)),
    'puresvd': PureSVDModel(factors=10),
    'ease': EASEModel(regularization=500),
}

# We will calculate several classic (precision@k and recall@k) and "beyond accuracy" metrics
metrics = {
    "prec@1": Precision(k=1),
    "prec@k": Precision(k=k),
    "recall@k": Recall(k=k),
    "novelty@k": MeanInvUserFreq(k=k),
    "serendipity@k": Serendipity(k=k),
    "ndcg@k": NDCG(k=k, log_base=3)
}

## TimeRangeSplitter

In [52]:
n_splits = 6

splitter = TimeRangeSplitter(
    test_size="30D",
    n_splits=n_splits,
    filter_already_seen=True,
    filter_cold_items=True,
    filter_cold_users=True,
)

In [42]:
splitter.get_test_fold_borders(dataset.interactions)

[(Timestamp('2025-04-15 00:00:00'), Timestamp('2025-05-15 00:00:00')),
 (Timestamp('2025-05-15 00:00:00'), Timestamp('2025-06-14 00:00:00')),
 (Timestamp('2025-06-14 00:00:00'), Timestamp('2025-07-14 00:00:00')),
 (Timestamp('2025-07-14 00:00:00'), Timestamp('2025-08-13 00:00:00')),
 (Timestamp('2025-08-13 00:00:00'), Timestamp('2025-09-12 00:00:00')),
 (Timestamp('2025-09-12 00:00:00'), Timestamp('2025-10-12 00:00:00'))]

In [53]:
%%time

# For each fold generate train and test part of dataset
# Then fit every model, generate recommendations and calculate metrics

cv_results = cross_validate(
    dataset=dataset,
    splitter=splitter,
    models=models,
    metrics=metrics,
    k=k,
    filter_viewed=True,
)

CPU times: user 4min 20s, sys: 36min 24s, total: 40min 44s
Wall time: 2min 28s


In [54]:
pd.DataFrame(cv_results["splits"])

,i_split,start,end,train,train_users,train_items,test,test_users,test_items
0,0,2025-04-15,2025-05-15,321926,138145,6952,4226,887,1205
1,1,2025-05-15,2025-06-14,340881,145027,7106,3442,812,1212
2,2,2025-06-14,2025-07-14,358458,153005,7248,3332,639,1377
3,3,2025-07-14,2025-08-13,372156,158313,7460,2879,611,1114
4,4,2025-08-13,2025-09-12,423317,196900,7715,2399,525,1063
5,5,2025-09-12,2025-10-12,435337,202580,7854,2339,481,988


In [55]:
pivot_results = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)
pivot_results.columns = pivot_results.columns.droplevel(1)

(
    pivot_results.style
    .set_caption("Mean values of metrics")
    .highlight_min(color='lightcoral', axis=0)
    .highlight_max(color='lightgreen', axis=0)
)

,prec@1,prec@k,recall@k,ndcg@k,novelty@k,serendipity@k
model,,,,,,
random,0.000952,0.000415,0.000404,0.000518,13.971305,0.000016
popular,0.010371,0.020163,0.032206,0.018465,5.313570,0.000004
most_rated,0.010801,0.019534,0.028722,0.018442,5.361707,0.000002
knn_tfidf_k=5,0.071840,0.034577,0.056755,0.041867,12.049068,0.000590
knn_bm25_k=5,0.078949,0.048027,0.086063,0.055176,7.732466,0.000507
als,0.028347,0.026184,0.042172,0.026825,7.369158,0.000127
brp,0.005640,0.001490,0.002166,0.002208,11.210951,0.000008
lightfm,0.002580,0.002802,0.003183,0.002847,10.861829,0.000027
puresvd,0.030100,0.023011,0.034762,0.024300,7.338402,0.000047


## LastNSplitter

In [56]:
n_splits = 6

splitter = LastNSplitter(
    5,
    n_splits=n_splits,
    filter_already_seen=True,
    filter_cold_items=True,
    filter_cold_users=True,
)

In [57]:
%%time

# For each fold generate train and test part of dataset
# Then fit every model, generate recommendations and calculate metrics

cv_results = cross_validate(
    dataset=dataset,
    splitter=splitter,
    models=models,
    metrics=metrics,
    k=k,
    filter_viewed=True,
)

CPU times: user 3min 19s, sys: 4min 41s, total: 8min
Wall time: 1min 26s


In [48]:
pd.DataFrame(cv_results["splits"])

,i_split,train,train_users,train_items,test,test_users,test_items
0,0,73283,1134,6064,4277,1032,1907
1,1,79694,1390,6144,5319,1274,2100
2,2,87679,1758,6229,6679,1607,2353
3,3,98290,2433,6324,9244,2217,2635
4,4,114150,3813,6444,14554,3444,3094
5,5,143774,8154,6595,30732,7366,3817


In [58]:
pivot_results = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)
pivot_results.columns = pivot_results.columns.droplevel(1)

(
    pivot_results.style
    .set_caption("Mean values of metrics")
    .highlight_min(color='lightcoral', axis=0)
    .highlight_max(color='lightgreen', axis=0)
)

,prec@1,prec@k,recall@k,ndcg@k,novelty@k,serendipity@k
model,,,,,,
random,0.000834,0.000862,0.000963,0.000836,8.994269,0.000015
popular,0.029800,0.026480,0.034692,0.027464,3.425563,0.000015
most_rated,0.028532,0.020401,0.026858,0.022019,3.814279,0.000001
knn_tfidf_k=5,0.058128,0.033074,0.046926,0.038295,7.486911,0.000423
knn_bm25_k=5,0.069389,0.044829,0.060791,0.050064,4.221615,0.000292
als,0.037590,0.027946,0.036324,0.029801,5.009720,0.000181
brp,0.014101,0.010974,0.013928,0.011345,4.764632,0.000036
lightfm,0.002690,0.006404,0.008382,0.005366,6.225514,0.000011
puresvd,0.035745,0.023039,0.031148,0.025755,4.319958,0.000023


## Анализ результатов

Интервал данных при запросе с Яндекс.Метрики был `2023-01-01 - 2025-10-12`, но после очистки записей без взаимодействия интервал времени стал `2023-09-19 - 2025-10-12` (сдвинулся аж до сентября (sic!)).

**Таблица метрик и лидеров не изменилась** при разных сплитах датасета (*по времени* и *по N=5 последних*)

- **Precision@1**, **Precision@5**, **Recall@5** и **NDCG@5** всех бьет - KNN (Imlicit Item KNN) с BM25, в качестве меры расстояния.

- **Novelty@5** - лучше всех Random (понятно почему). А вот KNN уступает почти всем, но...

- но нам скорее интересна метрика **Serendipity@5** - у KNN она очень хорошая, лучше всех с TFIDF мерой. При этом у всех она низкая, но у knn+tfidf, knn+bm25 и ease значительно выше остальных.

## Что за implicit и ImplicitItemKNN?
The implicit library is a fast Python package designed for building recommendation systems on implicit feedback datasets (like purchases, views, or play counts), where you don't have explicit ratings such as 1 to 5 stars. Its core functionality and the way you use its models are consistent.

The library provides several popular recommendation algorithms:

- **Alternating Least Squares** (ALS)
- **Bayesian Personalized Ranking** (BPR)
- **Item-Item Nearest Neighbour** models, which can use Cosine, TFIDF, or **BM25** as a distance metric.

## 🔍 Understanding BM25 and Item-Item Models

While the search results don't specify BM25Recommender, they confirm that implicit uses BM25 as one of the distance metrics within its **Item-Item Nearest Neighbour model**.

1. What is BM25?: BM25 is a renowned algorithm for lexical search, used to score and rank documents based on a query. It improves upon the classic TF-IDF method by considering factors like document length normalization
2. **How it works in recommendation**: In a collaborative filtering context, the "documents" are users, and the "words" are the items they have interacted with. An Item-Item model using BM25 would find items that are most similar to the items a user has already shown interest in, using the BM25 score to weight the importance of those interactions

In [60]:
# docs = users, words = items, that users iteracting

## Inference example

In [98]:
model = models['knn_bm25_k=5']

In [75]:
# https://gamersbase.store/game/endless-legend - Error 904
#df[df['item_id'].str.contains('Endless Legend')]

In [107]:
#df[df['item_id'].str.contains('Expeditions: Rome')]

In [109]:
#df[df['item_id'].str.contains('Hello Neighbor')]

In [105]:
v = [
    [0, 'Catizens', 1, '2023-12-24 00:00:00'], # indie cats strategy simulator https://gamersbase.store/game/catizens
    [0, 'EcoGnomix', 1, '2024-01-12 00:00:00'], # indie casual strategy simulator gnomes https://gamersbase.store/game/ecognomix
    [0, 'REKA', 1, '2024-02-02 00:00:00'], # indie adventure rpg casual magic atmosphere ru https://gamersbase.store/game/reka--4
    [0, 'Expeditions: Rome', 1, '2024-10-01 00:00:00'] # history rpg strategy rome https://gamersbase.store/game/expeditions-rome
]
df_v = pd.concat([df, pd.DataFrame(v, columns=columns)])
print(df_v.shape)
dataset_v = Dataset.construct(df_v)

(448534, 4)


In [101]:
model.fit(dataset_v)

In [110]:
%%time
recos = model.recommend(
    users=[0],
    dataset=dataset_v,
    k=k,
    filter_viewed=True,
)
recos

CPU times: user 40 ms, sys: 1.76 ms, total: 41.8 ms
Wall time: 39.9 ms


,user_id,item_id,score,rank
0,0.0,Hello Neighbor,1769.375244,1
1,0.0,Roblox Gift Card 20 EUR,1514.207764,2
2,0.0,Hello Neighbor VR: Search and Rescue,859.927856,3
3,0.0,ROBLOX GIFT CARD - 800 ROBUX,723.396423,4
4,0.0,RoboCop: Rogue City,644.698303,5


Hello Neighbor, конечно тоже indie, но в остальном и в целом это провал.

Надо учитывать платформы, чтобы для пк-бояр не рекомендовались роблокс и vr

Робокоп в конце вообще внезапно, конечно...

## Что дальше?

- надо добавить items features, все, что мы знаем про игры 
- надо определить weight взаимодействия пользователя с игрой. Подумать как определить weight из данных яндекс.метрики
- надо добавить users features, которые можно вычленить из данных яндекс.метрики
- пересчитать метрики на featured данных и проанализировать разницу
- потом можно попробовать трансформеры и сравнить с метриками на featured-данных

1. https://github.com/MobileTeleSystems/RecTools/blob/main/examples/4_dataset_with_features.ipynb
2. https://yandex.ru/dev/metrika/ru/logs/fields/visits
3. https://github.com/MobileTeleSystems/RecTools/tree/main/examples/tutorials

Параллельно, надо:

- подумать, как это можно **внедрить**
- непонятно, что делать с **играми, которых нет** (https://gamersbase.store/game/endless-legend)
- посмотреть лекции ВШЭ с курса по рексис
- изучить мануал "Debiased metrics calculation user guide" - https://github.com/MobileTeleSystems/RecTools/blob/main/examples/8_debiased_metrics.ipynb
